# Ingesting SharePoint Pages into Unity Catalog

This notebook pulls the content of SharePoint web pages (the `.aspx` pages users see in their browser) into a Unity Catalog Delta table. The output table feeds the same Knowledge Assistant processing pipeline as documents and lists.

Why a separate notebook for this: the Databricks Lakeflow Connect SharePoint connector only handles files in Document Libraries. It does not read SharePoint pages or web part content. The workaround is Microsoft Graph API, which exposes pages and their web parts directly. This notebook reuses the Entra ID app you set up in notebook 01.

The notebook is written for someone new to Databricks and Microsoft Graph. Every step is explicit.

### What you end up with

A Delta table containing one row per SharePoint page, with the page's title, URL, extracted text content, and raw web part metadata. From there:

- Knowledge Assistant can index the page text so it can answer questions about what users see on the SharePoint home page.
- Notebook 02's processing pipeline (chunking, metadata extraction, dedup) can be pointed at this table.
- Analytics on page content (most-edited pages, pages by topic, etc.) become possible via SQL queries.

## Prerequisites

Before starting, confirm:

- You completed notebook 01 (SharePoint connection setup). You have an Entra ID app registration with `Sites.Selected` permission and the app has been granted read access to the SharePoint sites you want to ingest.
- You have the Entra ID app's tenant ID, client ID, and client secret saved somewhere accessible.
- You have SharePoint admin permission to confirm the app has access to the specific site (or you completed Step 2.3 in notebook 01 already for this site).
- Your Databricks workspace has Unity Catalog enabled and serverless compute available.
- You have a destination catalog and schema where the output table will live, with `USE CATALOG`, `USE SCHEMA`, and `CREATE TABLE` privileges.

## Setup: collect the values you will paste in

The notebook needs five values. Collect them all up front so you can run cells without stopping.

### A. Entra ID credentials (from notebook 01)

From the Entra ID app registration you created in notebook 01, you need:

- **Tenant ID** (Directory ID). Find in Microsoft Entra ID > App registrations > your app > Overview > "Directory (tenant) ID".
- **Client ID** (Application ID). Same place: "Application (client) ID".
- **Client secret** (Value). The secret value you copied in notebook 01 Step 1.3. If you do not still have it, you can create a new one: go to your app > Certificates & secrets > New client secret. Remember to copy the "Value" column (NOT the "Secret ID" column).

### B. SharePoint site info

- **Tenant subdomain**. The first part of your SharePoint URL. If your SharePoint URL is `https://acme-corp.sharepoint.com/sites/data-governance`, your tenant subdomain is `acme-corp`.
- **Site name**. The path part after `/sites/`. From the example above, the site name is `data-governance`.

### C. Destination table

Decide on:

- **Catalog name** (e.g., `analytics_catalog`)
- **Schema name** (e.g., `bronze`)
- **Table name** (e.g., `sharepoint_pages`)

### Security note

The client secret is sensitive. The cell below uses direct paste for simplicity. After you run this notebook successfully, either delete the notebook or replace the direct-paste values with `dbutils.secrets.get(...)` calls. Do NOT commit a notebook containing a real client secret to a Git repository.

## Step 1: Get a Microsoft Graph access token

Microsoft Graph requires authentication on every API call. We use the OAuth client-credentials flow, which lets a service principal (your Entra ID app) authenticate without any user interaction.

Paste your five Setup values into the cell below and run it. A successful run prints `Got access token (expires in 3599 seconds)` or similar.

In [0]:
import requests

# Setup A values
TENANT_ID     = '<TENANT_ID>'      # e.g., abcdef12-3456-...
CLIENT_ID     = '<CLIENT_ID>'      # the Application (client) ID
CLIENT_SECRET = '<CLIENT_SECRET>'  # the Value from Certificates & secrets

# Setup B values
TENANT_SUBDOMAIN = '<TENANT_SUBDOMAIN>'  # e.g., acme-corp
SITE_NAME        = '<SITE_NAME>'         # e.g., data-governance

# Setup C values
CATALOG = '<CATALOG>'
SCHEMA  = '<SCHEMA>'
TABLE   = '<TABLE>'

# Get an access token
token_resp = requests.post(
    f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
    data={
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': 'https://graph.microsoft.com/.default',
        'grant_type': 'client_credentials',
    },
    timeout=30,
)

if token_resp.status_code != 200:
    raise RuntimeError(
        f'Failed to get access token. Status: {token_resp.status_code}. '
        f'Body: {token_resp.text[:500]}. '
        'Common causes: wrong TENANT_ID, wrong CLIENT_ID, expired or wrong CLIENT_SECRET. '
        'Double-check that you used the "Value" column not the "Secret ID" column.'
    )

access_token = token_resp.json()['access_token']
expires_in   = token_resp.json()['expires_in']
graph_headers = {'Authorization': f'Bearer {access_token}'}

print(f'Got access token (expires in {expires_in} seconds)')

## Step 2: Look up the SharePoint site ID

Microsoft Graph identifies sites by a long internal ID, not by the URL users see. Convert your tenant subdomain and site name into a site ID first.

In [0]:
# Look up the site ID from the tenant subdomain and site name
site_lookup_url = (
    f'https://graph.microsoft.com/v1.0/sites/'
    f'{TENANT_SUBDOMAIN}.sharepoint.com:/sites/{SITE_NAME}'
)
site_resp = requests.get(site_lookup_url, headers=graph_headers, timeout=30)

if site_resp.status_code == 404:
    raise RuntimeError(
        f'Site not found. Verify TENANT_SUBDOMAIN ({TENANT_SUBDOMAIN}) and '
        f'SITE_NAME ({SITE_NAME}) match an actual SharePoint site URL.'
    )
if site_resp.status_code == 403:
    raise RuntimeError(
        'Access denied (403). Two likely causes: (a) the Sites.Selected permission '
        'has not been granted by an admin, or (b) the app has not been granted '
        'read access to this specific site. Revisit notebook 01 Step 2.3 to grant '
        'the app read access to this site via Graph Explorer.'
    )
if site_resp.status_code != 200:
    raise RuntimeError(f'Site lookup failed. Status: {site_resp.status_code}. Body: {site_resp.text[:500]}')

SITE_ID = site_resp.json()['id']
print(f'Site ID: {SITE_ID}')
print(f'Site display name: {site_resp.json().get("displayName")}')

## Step 3: List all pages on the site

Pull the list of pages on the site. This does not yet fetch the web part content (the actual text on each page), it only lists the pages themselves.

In [0]:
# List all pages on the site
pages_url = f'https://graph.microsoft.com/v1.0/sites/{SITE_ID}/pages'
pages = []
next_url = pages_url

while next_url:
    resp = requests.get(next_url, headers=graph_headers, timeout=30)
    if resp.status_code != 200:
        raise RuntimeError(f'Failed to list pages. Status: {resp.status_code}. Body: {resp.text[:500]}')
    body = resp.json()
    pages.extend(body.get('value', []))
    next_url = body.get('@odata.nextLink')

print(f'Found {len(pages)} pages on the site.')
for p in pages[:10]:
    print(f'  - {p.get("title", "(no title)")} ({p.get("webUrl", "")})')
if len(pages) > 10:
    print(f'  ... and {len(pages) - 10} more')

## Step 4: Pull the content of each page

For each page found in Step 3, fetch its web parts and extract the text. The function below handles the three common web part types: text web parts, rich-text web parts, and standard web parts with embedded HTML.

In [0]:
import re
from datetime import datetime

def extract_text_from_webpart(wp):
    """Best-effort text extraction from a single web part."""
    # Text and rich-text web parts have innerHtml at the top level
    if 'innerHtml' in wp:
        return strip_html(wp['innerHtml'])
    # Standard web parts have innerHtml inside data
    data = wp.get('data', {})
    if isinstance(data, dict):
        parts = []
        for key in ('title', 'description', 'innerHtml'):
            value = data.get(key)
            if value and isinstance(value, str):
                parts.append(strip_html(value))
        return '\n'.join(parts)
    return ''


def strip_html(html):
    """Remove HTML tags and decode common entities. Lightweight, no BeautifulSoup."""
    text = re.sub(r'<[^>]+>', ' ', html)
    text = text.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def fetch_page_content(site_id, page_id):
    """Fetch web parts for a single page and concatenate their text."""
    url = (
        f'https://graph.microsoft.com/v1.0/sites/{site_id}/pages/'
        f'{page_id}/microsoft.graph.sitePage/webParts'
    )
    resp = requests.get(url, headers=graph_headers, timeout=30)
    if resp.status_code != 200:
        return '', []
    webparts = resp.json().get('value', [])
    chunks = []
    for wp in webparts:
        text = extract_text_from_webpart(wp)
        if text:
            chunks.append(text)
    return '\n\n'.join(chunks), webparts


# Pull content for all pages
ingested_at = datetime.utcnow()
rows = []
for i, page in enumerate(pages, 1):
    text, raw_webparts = fetch_page_content(SITE_ID, page['id'])
    rows.append({
        'page_id':       page['id'],
        'page_name':     page.get('name'),
        'page_title':    page.get('title'),
        'page_url':      page.get('webUrl'),
        'page_text':     text,
        'webpart_count': len(raw_webparts),
        'ingested_at':   ingested_at,
    })
    if i % 10 == 0:
        print(f'  Processed {i}/{len(pages)} pages')

print(f'Done. Extracted content from {len(rows)} pages.')

## Step 5: Write to a Delta table

Write the page content into Unity Catalog as a Delta table. The schema is intentionally simple. Notebook 02's processing pipeline can read directly from this table to chunk and index the content.

In [0]:
from pyspark.sql import Row

# Create the destination schema (idempotent)
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')

# Build DataFrame from the rows
df = spark.createDataFrame([Row(**r) for r in rows])

# Write to Delta, overwriting on each run (idempotent against re-ingestion)
table_fqn = f'{CATALOG}.{SCHEMA}.{TABLE}'
df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table_fqn)

# Enable Change Data Feed so downstream processing can pick up incremental changes
spark.sql(f"ALTER TABLE {table_fqn} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

print(f'Wrote {len(rows)} rows to {table_fqn}')

## Step 6: Verify what landed

In [0]:
display(spark.sql(f"""
  SELECT
    COUNT(*) AS page_count,
    SUM(CASE WHEN length(page_text) > 0 THEN 1 ELSE 0 END) AS pages_with_text,
    AVG(length(page_text)) AS avg_text_length,
    MIN(ingested_at) AS ingested_at
  FROM {CATALOG}.{SCHEMA}.{TABLE}
"""))

In [0]:
display(spark.sql(f"""
  SELECT page_title, page_url, webpart_count, length(page_text) AS text_length
  FROM {CATALOG}.{SCHEMA}.{TABLE}
  ORDER BY text_length DESC
  LIMIT 20
"""))

## Step 7: Hand off to the processing pipeline

The page content is now landed in Delta. Two ways to feed it into Knowledge Assistant:

### Option A: Point Knowledge Assistant at this table directly

In Agent Bricks, edit your Knowledge Assistant agent. Add `<CATALOG>.<SCHEMA>.<TABLE>` as an additional knowledge source. Map `page_text` as the content column. Knowledge Assistant will chunk and index the page text using its default chunking.

This is the fastest path. Use it if the default chunking is good enough for your use case.

### Option B: Run this table through the processing pipeline in notebook 02

If you need section-grouped chunking, metadata extraction, or deduplication, run the page content through notebook 02's pipeline. Adapt the Capability 6 (chunking) SQL in notebook 02 to read from the `<CATALOG>.<SCHEMA>.<TABLE>.page_text` column instead of the parsed document elements.

This gives you better retrieval quality at the cost of more pipeline complexity.

## Step 8: Schedule recurring refreshes

The notebook above runs once and writes the current snapshot of SharePoint pages to the Delta table. To keep the table fresh as the site changes, schedule the notebook to run on a recurring cadence using Lakeflow Jobs.

### Choosing a cadence

- **Daily**: the typical choice. SharePoint pages rarely change more than once a day, and 24-hour staleness is fine for most Knowledge Assistant corpora.
- **Hourly**: for high-velocity sites with frequent announcements or news updates.
- **Sub-hourly**: rarely needed. Microsoft Graph rate limits and SharePoint's own propagation latency make cadences below 15 minutes have diminishing returns.

### Step 8.1: Move credentials into Databricks Secrets

Before scheduling, replace the direct-paste credentials in Step 1 with `dbutils.secrets.get(...)` calls. Scheduled notebooks should never contain plaintext secrets, even in a private workspace.

First create a secret scope and add the three Entra ID values. Run these commands once from your local terminal (requires the Databricks CLI) or from any notebook with workspace admin access:

```bash
databricks secrets create-scope sharepoint-ingest
databricks secrets put-secret sharepoint-ingest tenant-id     --string-value '<TENANT_ID>'
databricks secrets put-secret sharepoint-ingest client-id     --string-value '<CLIENT_ID>'
databricks secrets put-secret sharepoint-ingest client-secret --string-value '<CLIENT_SECRET>'
```

Then edit Step 1's cell in this notebook to replace the placeholders with secret references:

```python
TENANT_ID     = dbutils.secrets.get(scope='sharepoint-ingest', key='tenant-id')
CLIENT_ID     = dbutils.secrets.get(scope='sharepoint-ingest', key='client-id')
CLIENT_SECRET = dbutils.secrets.get(scope='sharepoint-ingest', key='client-secret')
```

The other Setup values (tenant subdomain, site name, catalog, schema, table) are not sensitive and can stay as literal strings in the notebook.

### Step 8.2: Grant the Job's service principal access to the secret scope

When you schedule a notebook as a Lakeflow Job, Databricks runs it under a service principal identity, not under your personal user. That service principal must have `READ` permission on the secret scope, or the `dbutils.secrets.get(...)` calls will fail with a permission error.

From your local terminal or notebook:

```bash
databricks secrets put-acl sharepoint-ingest <service-principal-id> READ
```

If you do not yet have a dedicated service principal for ingestion jobs, the default behavior is that the Job runs as the user who created it (you). In that case you do not need this step until you switch to a dedicated SP, which is the recommended pattern for production.

### Step 8.3: Create the Lakeflow Job

1. In Databricks, click **Workflows** in the left sidebar (some workspaces label this as **Jobs & Pipelines**).
2. Click **Create job**.
3. Set:
   - **Name**: something descriptive like `Refresh SharePoint pages`
   - **Task name**: `ingest-pages`
   - **Type**: `Notebook`
   - **Source**: `Workspace`
   - **Path**: navigate to this notebook in your workspace
   - **Compute**: a serverless cluster is the simplest choice
4. In the right-hand panel, click **Add trigger** > **Scheduled**.
5. Set the cron schedule. Examples:
   - Daily at 6am: `0 0 6 * * ?`
   - Every hour: `0 0 * * * ?`
   - Every 15 minutes: `0 0/15 * * * ?`
6. Optionally add an email notification on task failure under **Notifications**.
7. Click **Create**.

The Job appears in your Workflows list. Click **Run now** to verify it works end-to-end on the schedule's compute. Once successful, the schedule takes over and the table refreshes automatically.

### How deletions are handled

The notebook uses `mode('overwrite')` when writing to Delta, so every run replaces the entire table contents with the current set of pages on the site. This means:

- Pages **added** to SharePoint show up in the table on the next run.
- Pages **edited** in SharePoint have their `page_text` updated in the table on the next run.
- Pages **deleted** from SharePoint disappear from the table on the next run.

Deletion handling is automatic with full refresh. No additional logic needed.

### Optional: incremental refresh for very large sites

If the site has tens of thousands of pages and full refresh becomes slow or expensive, switch to incremental mode using the Graph API's `lastModifiedDateTime` filter to pull only changed pages. This adds complexity: you need to track the last successful sync timestamp, query only modified pages, MERGE INTO the table instead of overwriting, and handle deletions explicitly by reconciling current page IDs against existing rows. Skip this unless you specifically hit scale issues with full refresh. For typical sites (hundreds of pages), full refresh runs in seconds.

### Downstream consumption

The Delta table has Change Data Feed enabled (set in Step 5). Notebook 02's processing pipeline can read the CDF feed to incrementally pick up only the pages that changed since its last run, avoiding re-chunking and re-indexing the entire corpus on every refresh. The combination is: notebook 04 refreshes the raw table daily, notebook 02 incrementally processes the changes, Knowledge Assistant indexes only what changed.

## Troubleshooting

### Step 1 errors (token acquisition)

- **Status 400 with "invalid_client"**: the CLIENT_ID or CLIENT_SECRET is wrong. Re-verify both. Most often, the CLIENT_SECRET is wrong because the user pasted the "Secret ID" instead of the "Value" column from Certificates & secrets.
- **Status 400 with "invalid_request" and AADSTS90002**: the TENANT_ID is wrong. Re-verify the Directory (tenant) ID from the Entra ID app's Overview page.
- **Status 401**: the client secret has expired. Generate a new one in Certificates & secrets and re-run.

### Step 2 errors (site lookup)

- **Status 404**: TENANT_SUBDOMAIN or SITE_NAME does not match a real SharePoint site URL. Verify by opening the site in a browser and inspecting the URL.
- **Status 403**: admin consent was not granted on `Sites.Selected`, OR the app has not been granted read access to this specific site via Graph Explorer. Revisit notebook 01 Steps 1.4 (admin consent) and 2.3 (per-site grant).

### Step 3 returns 0 pages

- The site has no published pages (only the default home page). Confirm by opening the site in SharePoint and checking the Pages library.
- The site is a Classic SharePoint site, not a Modern site. Classic sites do not expose pages through the v1.0 Graph API. There is no clean workaround other than migrating to Modern pages.

### Step 4 returns empty text for all pages

- The site uses third-party web parts that the simple extractor in this notebook does not understand. Inspect the `webpart_count` column to confirm web parts were found, then look at the raw web part response by adding `'raw_webparts': json.dumps(raw_webparts)` to each row dict in Step 4. From there, adapt the `extract_text_from_webpart` function to handle the specific web part types you see.

### Step 5 errors

- **Permission denied on the catalog or schema**: ask your Databricks workspace admin to grant you `USE CATALOG` on `<CATALOG>` and `CREATE TABLE` on `<SCHEMA>`.
- **Catalog or schema does not exist**: the cell creates the schema if missing, but it cannot create a catalog. Create the catalog manually via Catalog Explorer first.

## References

- [Microsoft Graph: List pages on a site](https://learn.microsoft.com/en-us/graph/api/sitepage-list)
- [Microsoft Graph: Get web parts of a page](https://learn.microsoft.com/en-us/graph/api/sitepage-list-webparts)
- [OAuth client credentials flow](https://learn.microsoft.com/en-us/azure/active-directory/develop/v2-oauth2-client-creds-grant-flow)
- [Sites.Selected permission](https://devblogs.microsoft.com/microsoft365dev/controlling-app-access-on-specific-sharepoint-site-collections/)
- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)